# Suite SVP — The SystemVerilog Patterns dialect

The patterns a `matches` compares a value with, binding its variables (IEEE 1800 12.6): the dialect
`Expressions.SystemVerilog.Patterns` of the SystemVerilog family, its kinds mbse-programs' patterns and a pattern's
parentheses, its `Term` listing them beside SystemVerilog's Expressions' branches, its holder, what a pattern binds, its
editions, its printing, and its evaluation, which refuses each kind by name (mbse-basis
[CAP-231: the SystemVerilog expression dialect, complete](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-231.md)).

In [ ]:
import { readFileSync } from "node:fs";

import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [SV, SVV, SVText] = [SystemVerilog.Expressions, SystemVerilog.Evaluators, SystemVerilog.Text];
const [SVP, SVPText] = [SVPatterns.Expressions, SVPatterns.Text];
const SVT = SVTypes.Expressions;

const store: any = new Builders.Store(); // the framework, the standard dialects SystemVerilog's types use, and its family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, SystemVerilog.install,
  SVTypes.install, SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install]) {
  store.transaction(install);
}
store.transaction(SystemVerilog.define); // SystemVerilog's types, which its evaluation reads
store.transaction(SVPatterns.define); // the editions the Patterns dialect covers
const Term: any = store.registered("Expressions.Framework.Term");

/** The holder of the term `spec` writes, `dialect`'s own (the Patterns dialect's unless given), written in a
 * transaction. */
function held(spec: unknown, dialect: F.Dialect = SVP.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

const pattern = (spec: unknown): any => F.term(held(spec)); // a term of the Patterns dialect, held by its holder
const sv = (spec: unknown): any => F.term(held(spec, SV.DIALECT)); // a term of SystemVerilog's Expressions, held by its holder
const spelled = (text: string) => (i: any) => i.spelling(text); // an identifier, of its spelling
const name = (text: string) => (u: any) => u.nameexpression((n: any) => n.name((i: any) => i.identifier(spelled(text))));
const variable = (text: string) => (u: any) => u.variablepattern((v: any) => v.name(spelled(text))); // `.text`
const wildcard = (u: any) => u.wildcardpattern((w: any) => w); // `.*`
const names = (set: ReadonlySet<string>): string[] => [...set].sort();

<a id="SVP-01"></a>
## SVP-01 · The Patterns dialect's union, its kinds' schemas in mbse-programs' shapes, the position that holds it, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
// Its kinds: mbse-programs' patterns, named and shaped as it has them, and a pattern's parentheses, kept as written; its
// Term lists them beside SystemVerilog's Expressions' branches, which a pattern may be (a constant pattern), with the
// Preprocessor's macro usage and invalid syntax; it has no helper of its own, its names Expressions' Identifier.
const union = (dialect: F.Dialect): string[] => store.registered(dialect.union).branches.map((b: any) => b.name); // its Term's branches
assert(same(SVP.TAGS, ["variablepattern", "wildcardpattern", "taggedpattern", "structurepattern", "patternmember",
  "parenthesizedpattern"]) && SVP.HELPERS.length === 0);
assert(same(union(SVP.DIALECT), [...SV.TAGS, "error", "missing", ...SVP.TAGS, "macrousage"])); // Expressions' installed first
assert(!SVP.TAGS.some((tag) => union(SV.DIALECT).includes(tag)) && same([...SVP.DIALECT.kinds().keys()], [...SVP.TAGS, "error", "missing"]));
assert(SVP.DIALECT.name() === "SystemVerilog Patterns" && SVP.DIALECT.tag === "systemverilog_patterns");
assert(SVP.DIALECT.path === "SystemVerilog.Patterns" && SVP.DIALECT.union === "Expressions.SystemVerilog.Patterns.Term");
const PatternsTerm: any = store.registered(SVP.DIALECT.union);
const kinds = new Map<string, any>(PatternsTerm.branches.map((b: any) => [b.name, b.schema]));
assert(same(SVP.TAGS.map((tag) => kinds.get(tag).name), ["VariablePattern", "WildcardPattern", "TaggedPattern", "StructurePattern",
  "PatternMember", "ParenthesizedPattern"].map((kind) => `Expressions.SystemVerilog.Patterns.${kind}`)));
assert(equal(new Map(SVP.TAGS.map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({ // each a value object, in mbse-programs' order
  variablepattern: ["name"], wildcardpattern: [], taggedpattern: ["member", "pattern"], structurepattern: ["items"],
  patternmember: ["name", "pattern"], parenthesizedpattern: ["pattern"] })) && !SVP.TAGS.some((t) => kinds.get(t).ref));
const Identifier: any = store.registered("Expressions.SystemVerilog.Expressions.Identifier"); // the family's one helper of each
assert(([["variablepattern", "name"], ["taggedpattern", "member"], ["patternmember", "name"]] as [string, string][]).every(
  ([tag, p]) => kinds.get(tag).properties.get(p).schema === Identifier));
assert(["taggedpattern", "patternmember", "parenthesizedpattern"].every((tag) => kinds.get(tag).properties.get("pattern").schema === PatternsTerm));
assert(kinds.get("structurepattern").properties.get("items").schema.element === PatternsTerm); // a pattern, an expression or a member
assert(equal(F._parts(kinds.get("structurepattern")), [[], ["items"], []]) && equal(F._parts(kinds.get("taggedpattern")), [["pattern"], [], ["member"]]));
// The position that holds its Term: a `matches`' pattern; its Term is a branch of the framework's, its dialect a singleton
const expressions = new Map<string, any>(store.registered(SV.DIALECT.union).branches.map((b: any) => [b.name, b.schema]));
assert(expressions.get("matchesexpression").properties.get("pattern").schema === PatternsTerm);
assert(Term.branches.find((b: any) => b.name === "systemverilog_patterns").schema === PatternsTerm);
assert(store.singleton("Expressions.SystemVerilog.Patterns") === store.Instances.Expressions.SystemVerilog.Patterns);
// Each dialect of the family reads its kinds, and it reads theirs, where its Term lists them.
assert(SV.DIALECT.kindNamed("variablepattern") === SVP.DIALECT.kinds().get("variablepattern"));
assert(SVP.DIALECT.kindNamed("binaryexpression") === SV.DIALECT.kinds().get("binaryexpression"));
assert(same(SVP.DIALECT.reads(), [SV.DIALECT, SVT.DIALECT, ...SV.DIALECT.reads().slice(2)]));
// Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
// validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
const Holder: any = store.registered("Expressions.SystemVerilog.Patterns.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === PatternsTerm && Holder.validate().length === 0);
const every = (u: any) => u.structurepattern((s: any) => s.items([ // '{a: .x, b: tagged Valid (.*), c: 1}
  (i: any) => i.patternmember((m: any) => m.name(spelled("a")).pattern(variable("x"))),
  (i: any) => i.patternmember((m: any) => m.name(spelled("b")).pattern((p: any) => p.taggedpattern((t: any) => t.member(spelled(
    "Valid")).pattern((q: any) => q.parenthesizedpattern((r: any) => r.pattern(wildcard)))))),
  (i: any) => i.patternmember((m: any) => m.name(spelled("c")).pattern(SV.literal(1n)))]));
const [holder, linked] = store.transaction(() => {
  const holder = store.Expressions.SystemVerilog.Patterns.Expression((e: any) => e.root(SVP.DIALECT.spec(every)));
  const linked = store.Schemas.OfExpression((s: any) => s.name("svp.every").expression((a: any) => a.insert((e: any) => e.expression(holder))));
  return [holder, linked];
});
assert(Validators.Validate(store)(Holder, holder).length === 0 && SVP.DIALECT.validate(holder).length === 0);
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
assert(F.dialectOf(F.term(holder)) === SVP.DIALECT.union && SVText.ToText(holder) === "'{a: .x, b: tagged Valid (.*), c: 1}");
for (const [write, read] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
  assert(F.same(read(Holder, write(Holder, holder)).root, holder.root));
  assert(F.same(read(PatternsTerm, write(PatternsTerm, F.term(holder))), F.term(holder)));
}
// The schema refuses a pattern where a union that lacks it is held: an Expressions holder's root.
raises(Error, () => store.transaction(() => store.Expressions.SystemVerilog.Expressions.Expression(
  (e: any) => e.root(SVP.DIALECT.make(new F.Form("wildcardpattern"))))), "'wildcardpattern' is not a member of a union value");
// Another family's term is refused, as before.
raises(ValueError, () => SVP.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1n, Patterns.Expressions.variable("a")),
  Patterns.Expressions.DIALECT))), "a Patterns let is not a SystemVerilog Patterns term");

<a id="SVP-02"></a>
## SVP-02 · Each kind and its checks: what it requires, its identifiers, a structure pattern's members in order or by name; forms and `make`, and inference

In [ ]:
// Each kind's checks: what it requires, its identifiers' spellings, and mbse-programs' check that a structure pattern's
// members are in order or by name, not both; a tagged pattern's pattern is optional (`tagged None`). A constant pattern
// is the expression it is, its names references the scope must bind.
const validate = (spec: unknown, bound: string[] = []): string[] => SVP.DIALECT.validate(pattern(spec), bound);
const member = (text: string, inner: unknown) => (i: any) => i.patternmember((m: any) => m.name(spelled(text)).pattern(inner));
for (const [spec, problems] of [
  [variable("a"), []], [variable("\\a+b"), []], [wildcard, []],
  [(u: any) => u.variablepattern((v: any) => v), ["a variablepattern needs a name"]],
  [variable("1a"), ["'1a' is not an identifier"]], [variable("$a"), ["'$a' is not an identifier"]],
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("None"))), []], // no pattern: `tagged None`
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("Valid")).pattern(variable("a"))), []],
  [(u: any) => u.taggedpattern((t: any) => t.pattern(wildcard)), ["a taggedpattern needs a member"]],
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("9v")).pattern(variable("a"))), ["'9v' is not an identifier"]],
  [(u: any) => u.structurepattern((s: any) => s.items([variable("a"), wildcard, SV.literal(1n)])), []], // in order
  [(u: any) => u.structurepattern((s: any) => s.items([member("a", variable("x")), member("b", SV.literal(1n))])), []], // by name
  [(u: any) => u.structurepattern((s: any) => s.items([member("a", variable("x")), variable("y")])),
    ["a StructurePattern's members are in order or by name, not both"]],
  [(u: any) => u.structurepattern((s: any) => s.items([(i: any) => i.patternmember((m: any) => m.name(spelled("a")))])),
    ["argument 0: a patternmember needs a pattern"]],
  [(u: any) => u.structurepattern((s: any) => s.items([member("2a", variable("a"))])), ["argument 0: '2a' is not an identifier"]],
  [(u: any) => u.patternmember((m: any) => m.pattern(wildcard)), ["a patternmember needs a name"]],
  [(u: any) => u.parenthesizedpattern((p: any) => p.pattern(variable("a"))), []],
  [(u: any) => u.parenthesizedpattern((p: any) => p), ["a parenthesizedpattern needs a pattern"]],
  [(u: any) => u.parenthesizedpattern((p: any) => p.pattern(variable("1a"))), ["pattern: '1a' is not an identifier"]],
  [name("x"), ["nameexpression 'x' is not bound"]], // a constant pattern: an expression
  [(u: any) => u.structurepattern((s: any) => s.items([SV.binary("+", name("x"), 1n)])), ["argument 0: left: nameexpression 'x' is not bound"]],
] as [unknown, string[]][]) {
  assert(same(validate(spec), problems), JSON_(problems, validate(spec)));
}
function JSON_(...values: unknown[]): string { return globalThis.JSON.stringify(values); }
assert(validate(name("x"), ["x"]).length === 0 && SVP.problems(F.node("wildcardpattern", F.kindOf(pattern(wildcard))[1])).length === 0);
// Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
// from which make writes it again; a variable's name is the name the framework reads it by.
assert(SVP.TAGS.every((tag) => SVP.DIALECT.kinds().get(tag)!.role === F.APPLICATION));
const tree = pattern((u: any) => u.structurepattern((s: any) => s.items([ // '{a: .x, b: tagged V ('{.*, 1}), c: tagged None}
  member("a", variable("x")), member("b", (p: any) => p.taggedpattern((t: any) => t.member(spelled("V")).pattern(
    (q: any) => q.parenthesizedpattern((r: any) => r.pattern((w: any) => w.structurepattern((s2: any) => s2.items([wildcard, SV.literal(1n)]))))))),
  member("c", (p: any) => p.taggedpattern((t: any) => t.member(spelled("None"))))])));
const walked = [...F.walk(tree)];
assert(same(names(new Set(walked.map((node) => F.kindOf(node)[0]))), names(new Set([...SVP.TAGS, "literal"]))));
for (const node of walked) {
  const made = pattern(SVP.DIALECT.make(F.form(node)));
  assert(F.same(made, node) && made !== node);
}
const formed = F.form(pattern(variable("x"))), part = formed.attributes.get("name") as F.Part;
assert(formed.kind === "variablepattern" && part.kind === "identifier" && equal(part.attributes, map({ spelling: "x" })) && part.size === 0);
assert(same(F.form(pattern((u: any) => u.taggedpattern((t: any) => t.member(spelled("None"))))).arguments, [null])); // its empty slot
assert(F.nameOf(pattern(variable("x"))) === "x" && F.nameOf(pattern(wildcard)) === null);
assert(SVP.DIALECT.infer(tree) === D.Anything && SVP.DIALECT.infer(pattern(variable("x"))) === D.Anything);
assert(SVP.DIALECT.infer(pattern(SV.binary("+", name("x"), 1n)), { x: D.Anything }) === D.Anything);

<a id="SVP-03"></a>
## SVP-03 · What a pattern binds: its variables, wherever they nest, which a `matches` binds in its conditional's consequence and the conditions after it in a `&&&`

In [ ]:
// What a pattern binds (`boundNames`): its variables, wherever its tagged, structure, member and parenthesized patterns
// nest them; a wildcard, a constant pattern (an expression, whose names are references), invalid syntax, and a variable
// with no name or no spelling bind none.
const binds = (spec: unknown): string[] => names(SVP.boundNames(pattern(spec)));
assert(same(binds(variable("a")), ["a"]) && binds(wildcard).length === 0 && binds(name("a")).length === 0);
assert(binds((u: any) => u.taggedpattern((t: any) => t.member(spelled("None")))).length === 0);
assert(same(binds((u: any) => u.taggedpattern((t: any) => t.member(spelled("V")).pattern(variable("a")))), ["a"]));
assert(same(binds((u: any) => u.parenthesizedpattern((p: any) => p.pattern(variable("a")))), ["a"]));
assert(binds((u: any) => u.parenthesizedpattern((p: any) => p)).length === 0);
assert(same(binds((u: any) => u.patternmember((m: any) => m.name(spelled("m")).pattern(variable("a")))), ["a"])); // not its member's name
assert(same(binds((u: any) => u.structurepattern((s: any) => s.items([variable("a"), (i: any) => i.structurepattern((t: any) => t.items([
  variable("b"), wildcard])), (i: any) => i.taggedpattern((t: any) => t.member(spelled("V")).pattern(variable("c"))), name("d")]))), ["a", "b", "c"]));
assert(binds((u: any) => u.structurepattern((s: any) => s.items([]))).length === 0);
assert(binds((u: any) => u.variablepattern((v: any) => v)).length === 0 && binds((u: any) => u.variablepattern((v: any) => v.name((i: any) => i))).length === 0);
assert(binds((u: any) => u.error((e: any) => e.text(".a +").terms([SVP.DIALECT.spec(variable("a"))]))).length === 0); // what invalid syntax holds
// A pattern binds, but is no binder of its own: none of its kinds refines Let, and none has scopes; the `matches` that
// holds it binds its variables in its conditional's consequence and the conditions after it in a `&&&` (12.6.2, 12.6.3),
// not in the value it matches nor in the alternative.
assert(!SVP.TAGS.some((tag) => F.isBinder(kinds.get(tag)) || SVP.DIALECT.kinds().get(tag)!.scopes !== null));
const matches = (value: unknown, inner: unknown) => (u: any) => u.matchesexpression((m: any) => m.value(value).pattern(inner));
const pair = (u: any) => u.structurepattern((s: any) => s.items([variable("a"), variable("b")])); // '{.a, .b}
const chosen = sv((u: any) => u.conditionalexpression((c: any) => c.condition(matches(name("v"), pair)).consequence(
  SV.binary("+", name("a"), name("b"))).alternative(name("a"))));
assert(same(names(SY.free(chosen, SV.DIALECT)), ["a", "v"])); // the alternative's `a` is free
assert(same(SV.DIALECT.validate(chosen, ["v"]), ["alternative: nameexpression 'a' is not bound"]));
assert(equal(SV.DIALECT.kinds().get("conditionalexpression")!.scopes!(F.kindOf(chosen)[1]).map(names), [[], ["a", "b"], []]));
assert(same(names(SY.free(sv(matches(name("a"), variable("a"))), SV.DIALECT)), ["a"])); // the value it matches is outside it
const both = sv((u: any) => u.conditionalexpression((c: any) => c.condition((p: any) => p.predicateexpression((q: any) => q.conditions([
  matches(name("v"), variable("a")), SV.binary(">", name("a"), 0n),
  matches(name("w"), (t: any) => t.taggedpattern((g: any) => g.member(spelled("T")).pattern(variable("b")))),
  SV.binary("<", name("b"), name("a"))]))).consequence(SV.binary("+", name("a"), name("b"))).alternative(SV.literal(0n))));
assert(same(names(SY.free(both, SV.DIALECT)), ["v", "w"]) && SV.DIALECT.validate(both, ["v", "w"]).length === 0);
const predicate = F.form(both).arguments[0];
assert(equal(SV.DIALECT.kinds().get("predicateexpression")!.scopes!(F.kindOf(predicate)[1]).map(names), [
  [], ["a"], ["a"], ["a", "b"]])); // each condition: what those before it matched

<a id="SVP-04"></a>
## SVP-04 · A tree mixing patterns with expressions and types, validated and printed from the Patterns dialect's holder and from an Expressions holder through a `matches`

In [ ]:
// A tree mixing patterns with expressions and types: a pattern's constant (an expression, a cast of a Types term) nested
// in a structure pattern, validated and printed from the Patterns dialect's holder, and from an Expressions holder
// through a `matches`, the position that holds the Patterns dialect's Term.
const mixed = (u: any) => u.structurepattern((s: any) => s.items([
  member("a", variable("x")),
  member("b", (p: any) => p.taggedpattern((t: any) => t.member(spelled("Valid")).pattern(
    (q: any) => q.structurepattern((r: any) => r.items([variable("y"), wildcard]))))),
  member("c", SV.cast((t: any) => t.integeratomtype((a: any) => a.keyword("int")), SV.binary("+", name("w"), 1n))),
  member("d", (p: any) => p.parenthesizedpattern((q: any) => q.pattern(variable("z"))))]));
const SOURCE = "'{a: .x, b: tagged Valid '{.y, .*}, c: int'(w + 1), d: (.z)}";
const own = pattern(mixed);
assert(SVPText.ToText(own) === SOURCE && SVText.ToText(own) === SOURCE && SVPText.ToText === SVText.ToText);
assert(same(SVP.DIALECT.validate(own), ["argument 2: pattern: value: left: nameexpression 'w' is not bound"]));
assert(SVP.DIALECT.validate(own, ["w"]).length === 0 && same(names(SY.free(own, SVP.DIALECT)), ["w"]) && same(names(SVP.boundNames(own)), ["x", "y", "z"]));
const castPattern = F.form(F.form(own).arguments[2]).arguments[0]; // c's pattern: a cast, its type a Types term
assert(F.dialectOf(castPattern) === SVP.DIALECT.union && F.dialectOf(F.form(castPattern).arguments[0]) === SVT.DIALECT.union);
const chosenMixed = sv((u: any) => u.conditionalexpression((c: any) => c.condition((m: any) => m.matchesexpression((q: any) => q.value(name("v")).pattern(mixed)))
  .consequence(SV.binary("+", SV.binary("+", name("x"), name("y")), name("z"))).alternative(name("w"))));
assert(SVText.ToText(chosenMixed) === `v matches ${SOURCE} ? x + y + z : w`);
assert(SV.DIALECT.validate(chosenMixed, ["v", "w"]).length === 0 && same(names(SY.free(chosenMixed, SV.DIALECT)), ["v", "w"]));
assert(same(SV.DIALECT.validate(chosenMixed), ["condition: value: nameexpression 'v' is not bound",
  "condition: pattern: argument 2: pattern: value: left: nameexpression 'w' is not bound",
  "alternative: nameexpression 'w' is not bound"]));
assert(F.dialectOf(F.form(F.form(chosenMixed).arguments[0]).arguments[1]) === SVP.DIALECT.union); // the matches' pattern
assert(SV.DIALECT.infer(chosenMixed, { v: D.Anything, w: D.Anything }) === D.Anything);
assert(SVText.ToText(sv((u: any) => u.predicateexpression((p: any) => p.conditions([
  (c: any) => c.matchesexpression((m: any) => m.value(name("v")).pattern(variable("a"))), SV.binary(">", name("a"), 0n)])))) === "v matches .a &&& a > 0");

<a id="SVP-05"></a>
## SVP-05 · Invalid syntax where a pattern stands: validated, printed as written, and refused by evaluation

In [ ]:
// Invalid syntax: the Patterns dialect's Term holds `error` and `missing`, where a pattern stands; validation reports
// them, printers write them as written, and evaluation refuses them, as every dialect's.
const broken = pattern((u: any) => u.structurepattern((s: any) => s.items([
  (i: any) => i.error((e: any) => e.text(".a +").terms([SVP.DIALECT.spec(variable("a"))])), (i: any) => i.missing((m: any) => m.kind("pattern"))])));
assert(same(SVP.DIALECT.validate(broken), ["argument 0: invalid syntax '.a +'", "argument 1: missing pattern"]));
assert(SVText.ToText(broken) === "'{.a +, }" && SVP.boundNames(broken).size === 0);
assert(same(F.form(broken).arguments.map((a) => SVP.DIALECT.kind(a)[2].role), [F.INVALID, F.INVALID]));
const taggedMissing = pattern((u: any) => u.taggedpattern((t: any) => t.member(spelled("V")).pattern((p: any) => p.missing((m: any) => m))));
assert(same(SVP.DIALECT.validate(taggedMissing), ["pattern: missing a term"]) && SVText.ToText(taggedMissing) === "tagged V ");
raises(ValueError, () => SVText.ToText(pattern((u: any) => u.parenthesizedpattern((p: any) => p.pattern((e: any) => e.error((r: any) => r))))),
  "SystemVerilog Patterns error has no text to print");
raises(ValueError, () => SVV.OfAny(broken), "cannot evaluate an invalid expression: argument 0: invalid syntax '.a +'");

<a id="SVP-06"></a>
## SVP-06 · Its editions as data: IEEE 1800-2005 to 1800-2023, each with every one of its own kinds

In [ ]:
// Its editions, as data: its define covers IEEE 1800's five, owned by their designations, each found by its
// designation or its common name, each with every one of its own kinds (all of them since IEEE 1800-2005, as
// mbse-programs' SINCE has them) and none of the kinds it reads; run again, it writes nothing.
const patterns_ = store.Instances.Expressions.SystemVerilog.Patterns;
const IEEE1800 = [2005, 2009, 2012, 2017, 2023].map((year) => `IEEE Std 1800-${year}`);
assert(same([...FS.standards(patterns_).keys()], IEEE1800) && equal(SVP.SINCE, new Map(SVP.TAGS.map((tag) => [tag, 3]))));
assert(SV.EDITIONS[3]![0] === IEEE1800[0]); // the fourth edition of the family's, its first
assert(same(FS.tags(FS.find(patterns_, "SystemVerilog-2005")!), ["IEEE Std 1800-2005", "SystemVerilog-2005"]));
assert(FS.find(patterns_, "SystemVerilog-2023") === FS.find(patterns_, "IEEE Std 1800-2023") && FS.find(patterns_, "Verilog-2005") === null);
assert(equal(FS.properties(FS.find(patterns_, "SystemVerilog-2023")!), map({ bodies: ["org.ieee"], number: "1800",
  revision: ["year", 2023n], names: ["SystemVerilog-2023"] })));
assert([...FS.standards(patterns_).values()].every((edition) => same(FS.kinds(edition), SVP.TAGS.map((tag) => kinds.get(tag)))));
assert(SVP.TAGS.every((tag) => same([...FS.of(patterns_, kinds.get(tag)).keys()], IEEE1800)));
assert(FS.of(patterns_, kinds.get("literal")).size === 0 && FS.of(patterns_, store.registered("Expressions.Framework.Error")).size === 0);
assert(store.transaction(SVPatterns.define)[0].get("SystemVerilog_2023") === FS.find(patterns_, "SystemVerilog-2023"));
assert(same([...FS.standards(patterns_).keys()], IEEE1800)); // nothing written again

<a id="SVP-07"></a>
## SVP-07 · Printing every kind as mbse-programs writes patterns

In [ ]:
// Printing, as mbse-programs' printer writes patterns (`pattern_text`): a variable `.name`, a wildcard `.*`, a tagged
// pattern `tagged member pattern`, a structure pattern in order or by name, a member `name: pattern`, parentheses as
// written, and a constant pattern as the expression it is, from the Patterns dialect's holder.
const printed: [unknown, string][] = [
  [variable("a"), ".a"], [variable("\\a+b"), ".\\a+b"], [wildcard, ".*"],
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("None"))), "tagged None"],
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("Valid")).pattern(variable("a"))), "tagged Valid .a"],
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("T")).pattern((p: any) => p.taggedpattern((q: any) => q.member(spelled("U")).pattern(wildcard)))),
    "tagged T tagged U .*"],
  [(u: any) => u.structurepattern((s: any) => s.items([])), "'{}"],
  [(u: any) => u.structurepattern((s: any) => s.items([variable("a"), SV.literal(1n), SV.binary("+", name("x"), 1n)])), "'{.a, 1, x + 1}"],
  [(u: any) => u.structurepattern((s: any) => s.items([member("a", variable("x")), member("b", (p: any) => p.taggedpattern(
    (t: any) => t.member(spelled("V"))))])), "'{a: .x, b: tagged V}"],
  [(u: any) => u.patternmember((m: any) => m.name(spelled("a")).pattern(wildcard)), "a: .*"],
  [(u: any) => u.parenthesizedpattern((p: any) => p.pattern((q: any) => q.taggedpattern((t: any) => t.member(spelled("V")).pattern(variable("a"))))),
    "(tagged V .a)"],
  [SV.conditional(name("x"), 1n, 2n), "x ? 1 : 2"], // a constant pattern, unparenthesized as mbse-programs writes it
];
for (const [spec, source] of printed) assert(SVText.ToText(pattern(spec)) === source, `${source} ${SVText.ToText(pattern(spec))}`);
const printedKinds = new Set(printed.map(([spec]) => F.kindOf(pattern(spec))[0]));
assert(SVP.TAGS.every((tag) => printedKinds.has(tag))); // every kind
assert(same([...SVPText.TEXTS.keys()], [...SVP.TAGS]) && SVPText.PRECEDENCE.size === 0 && SVPText.INLINE.size === 0);

<a id="SVP-08"></a>
## SVP-08 · Evaluation refuses each of its kinds by name, and a `matches` before a conditional's branches are sized

In [ ]:
// Evaluation (Q11): SystemVerilog's evaluator evaluates none of its kinds, and refuses each by name; the `matches`
// that holds them is refused by name too.
const refused: [unknown, string][] = [
  [variable("a"), "variablepattern"], [wildcard, "wildcardpattern"],
  [(u: any) => u.taggedpattern((t: any) => t.member(spelled("None"))), "taggedpattern"],
  [(u: any) => u.structurepattern((s: any) => s.items([wildcard])), "structurepattern"],
  [(u: any) => u.patternmember((m: any) => m.name(spelled("a")).pattern(wildcard)), "patternmember"],
  [(u: any) => u.parenthesizedpattern((p: any) => p.pattern(wildcard)), "parenthesizedpattern"],
];
assert(same(refused.map(([, tag]) => tag), [...SVP.TAGS]));
for (const [spec, tag] of refused) raises(NotImplementedError, () => SVV.OfAny(held(spec)), `SystemVerilog Patterns ${tag} is not evaluated`);
raises(NotImplementedError, () => SVV.OfAny(held((u: any) => u.matchesexpression((m: any) => m.value(name("v")).pattern(variable("a"))), SV.DIALECT),
  { v: 1n }), "SystemVerilog matchesexpression is not evaluated");
// A conditional's condition is sized first, so its `matches` is refused by name before its consequence, which names what
// the pattern binds, is read: `v matches .a ? a : 0`, not "a" unbound.
const vMatches = (u: any) => u.matchesexpression((m: any) => m.value(name("v")).pattern(variable("a")));
raises(NotImplementedError, () => SVV.OfAny(held(SV.conditional(vMatches, name("a"), 0n), SV.DIALECT), { v: 1n }),
  "SystemVerilog matchesexpression is not evaluated");

<a id="SVP-09"></a>
## SVP-09 · Complete against slang: each of slang's pattern syntax kinds is one of its kinds

In [ ]:
// Complete against slang: each of slang's syntax kinds that is a pattern (12.6) maps to one of its kinds, and each of
// its kinds is mapped from one (completeness data, Q9); an ordered member and an expression pattern are the patterns
// and expressions they hold.
const grammar = globalThis.JSON.parse(readFileSync("../../grammars/systemverilog-slang.json", "utf8"));
const mappedPatterns = new Map<string, string>();
for (const node of grammar.nodes) {
  for (const item of [...(node.kinds ?? []), ...(node.helpers ?? [])]) if (item.startsWith("Patterns.")) mappedPatterns.set(node.node, item.slice("Patterns.".length));
}
assert(equal(mappedPatterns, map({ NamedStructurePatternMember: "patternmember", ParenthesizedPattern: "parenthesizedpattern",
  StructurePattern: "structurepattern", TaggedPattern: "taggedpattern", VariablePattern: "variablepattern",
  WildcardPattern: "wildcardpattern" })));
assert(same([...new Set(mappedPatterns.values())].sort(), [...SVP.TAGS, ...SVP.HELPERS].sort()));
assert(grammar.nodes.filter((node: any) => ["OrderedStructurePatternMember", "ExpressionPattern"].includes(node.node)).every((node: any) => "not" in node));